In [5]:
import configparser
import os
import boto3

CREDENTIALS_PATH = "../.aws/credentials"

if not os.path.exists(CREDENTIALS_PATH):
    raise FileNotFoundError(f"No se encontró el archivo de credenciales en: {os.path.abspath(CREDENTIALS_PATH)}")


AWS_REGION = "us-east-1"

aws_config = configparser.ConfigParser()
aws_config.read(CREDENTIALS_PATH)

PROFILE = aws_config.sections()[0]

AWS_ACCESS_KEY = aws_config[PROFILE].get("aws_access_key_id")
AWS_SECRET_KEY = aws_config[PROFILE].get("aws_secret_access_key")
AWS_SESSION_TOKEN = aws_config[PROFILE].get("aws_session_token", None)

In [6]:
def crear_cliente(servicio):
    return boto3.client(
        servicio,
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY,
        aws_session_token=AWS_SESSION_TOKEN,
        region_name=AWS_REGION
    )

Probar que las credenciales funcionan:

In [7]:
sts = crear_cliente("sts")
print(sts.get_caller_identity())

{'UserId': 'AROAXF33W5AWU2HEZBHCE:aitaguduq@alu.edu.gva.es', 'Account': '493643819053', 'Arn': 'arn:aws:sts::493643819053:assumed-role/AWSReservedSSO_AWSAdministratorAccess_e1d3988c982ae3e1/aitaguduq@alu.edu.gva.es', 'ResponseMetadata': {'RequestId': '544ed8f2-e718-4ad1-af69-cb2584e1a40a', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '544ed8f2-e718-4ad1-af69-cb2584e1a40a', 'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6UzoxNzc4MTA5MDYzNjMxOlI6eHp0VEVFU2c=', 'content-type': 'text/xml', 'content-length': '509', 'date': 'Wed, 06 May 2026 23:11:03 GMT'}, 'RetryAttempts': 0}}


Comprobar que se reconocen correctamente las matriculas:

In [8]:
import boto3
import re

rekognition = crear_cliente("rekognition")

def detectar_matriculas(image_path):
    with open(image_path, 'rb') as image_file:
        image_bytes = image_file.read()

    response = rekognition.detect_text(
        Image={'Bytes': image_bytes}
    )

    textos = response['TextDetections']

    resultados = []

    for t in textos:
        if t["Type"] == "LINE":
            texto = t["DetectedText"].upper().replace(" ", "").replace("-", "")
            confianza = t["Confidence"]

            print(f"{texto} - {confianza}")

            resultados.append(texto)

    return resultados


In [19]:
imagen = "../assets/coche.jpg"
matriculas = detectar_matriculas(imagen)

WIOC748 - 81.10054016113281


In [ ]:
dynamodb = boto3.resource("dynamodb", region_name=REGION)
client = crear_cliente("dynamodb")


def crear_tabla_si_no_existe():
    try:
        client.describe_table(TableName=TABLE_NAME)
        print("✔ La tabla ya existe")
    except client.exceptions.ResourceNotFoundException:
        print("📦 Creando tabla...")

        client.create_table(
            TableName=TABLE_NAME,
            KeySchema=[
                {"AttributeName": "matricula", "KeyType": "HASH"}
            ],
            AttributeDefinitions=[
                {"AttributeName": "matricula", "AttributeType": "S"}
            ],
            BillingMode="PAY_PER_REQUEST"
        )

        # esperar a que esté activa
        waiter = client.get_waiter("table_exists")
        waiter.wait(TableName=TABLE_NAME)

        print("✔ Tabla creada correctamente")


# 2. Guardar matrícula
def guardar_matricula(matricula):
    table = dynamodb.Table(TABLE_NAME)

    table.put_item(
        Item={
            "matricula": matricula
        }
    )

    print(f"✔ Guardada: {matricula}")

In [9]:
# =========================
# PRUEBA SIMPLE AWS
# =========================

try:
    response = rekognition.list_collections()

    print("✅ AWS Rekognition funciona correctamente")
    print(response)

except Exception as e:

    import traceback

    print("❌ ERROR AWS")
    print(type(e))
    print(e)

    traceback.print_exc()

✅ AWS Rekognition funciona correctamente
{'CollectionIds': [], 'FaceModelVersions': [], 'ResponseMetadata': {'RequestId': '8e1a98ca-783b-4fc6-b570-85eb40218b25', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '8e1a98ca-783b-4fc6-b570-85eb40218b25', 'content-type': 'application/x-amz-json-1.1', 'content-length': '43', 'date': 'Wed, 06 May 2026 23:41:39 GMT'}, 'RetryAttempts': 0}}


In [13]:
import re

IMAGE_PATH = "../assets/matriculaCoche.jpg"

with open(IMAGE_PATH, "rb") as image:

    image_bytes = image.read()

try:

    response = rekognition.detect_text(
        Image={
            'Bytes': image_bytes
        }
    )

    print("✅ OCR funcionando")

    texto_total = ""

    for texto in response["TextDetections"]:

        if texto["Type"] == "LINE":

            detected = texto["DetectedText"]

            print("Texto detectado:", detected)

            texto_total += detected + " "

    # Regex matrícula española
    patron = r'\d{4}[BCDFGHJKLMNPRSTVWXYZ]{3}'

    match = re.search(
        patron,
        texto_total.replace(" ", "").upper()
    )

    if match:
        print("✅ Matrícula encontrada:", match.group())
    else:
        print("❌ No se detectó matrícula")

except Exception as e:

    import traceback

    print("❌ ERROR OCR")
    print(type(e))
    print(e)

    traceback.print_exc()

❌ ERROR OCR
<class 'botocore.errorfactory.InvalidImageFormatException'>
An error occurred (InvalidImageFormatException) when calling the DetectText operation: Request has invalid image format


Traceback (most recent call last):
  File "C:\Users\Aitor\AppData\Local\Temp\ipykernel_13152\3633301789.py", line 11, in <module>
    response = rekognition.detect_text(
        Image={
            'Bytes': image_bytes
        }
    )
  File "c:\Users\Aitor\anaconda3\Lib\site-packages\botocore\client.py", line 606, in _api_call
    return self._make_api_call(operation_name, kwargs)
           ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Aitor\anaconda3\Lib\site-packages\botocore\context.py", line 123, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\Aitor\anaconda3\Lib\site-packages\botocore\client.py", line 1094, in _make_api_call
    raise error_class(parsed_response, operation_name)
botocore.errorfactory.InvalidImageFormatException: An error occurred (InvalidImageFormatException) when calling the DetectText operation: Request has invalid image format
